# Data Cleaning and Preprocessing of a Large Dataset Using PySpark

**Course:** CS 405 - Large Scale Data Engineering  
**Assignment Title:** Data Cleaning and Preprocessing of a Large Dataset Using PySpark  
**Date:** June 3, 2026  
**Submitted By:** [Student Name / Placeholder]  

---

## 1. Project Introduction

In modern data engineering, raw data is rarely ready for direct consumption. It is typically riddled with missing values, duplicate entries, inconsistent text formatting, and incorrect data types. Before applying any machine learning algorithms, performing analytics, or writing reports, the dataset must go through a comprehensive cleaning and preprocessing pipeline.

While standard Python libraries like Pandas are ideal for small to medium datasets that easily fit into a single machine's RAM, they hit scaling bottlenecks when processing gigabytes or terabytes of information. **Apache Spark** is an open-source, distributed computing system designed for speed, scalability, and ease of use when handling massive datasets. Using **PySpark** (the Python API for Apache Spark), we can distribute cleaning computations across a cluster of machines utilizing a resilient, distributed, and lazy execution model.

### Assignment Objectives:
1. **Environment Setup**: Initialize a local Spark Session.
2. **Data Ingestion**: Load a dirty `data.csv` dataset, maintaining columns as strings initially.
3. **Exploration**: Inspect schema, shape, and sample records.
4. **Missing Value Treatment**: Detect null counts and perform statistical imputations (median/mean for numericals, mode/placeholders for categoricals).
5. **Deduplication**: Identify and remove identical and redundant records, verifying the counts.
6. **Data Type Correction**: Cast all fields to correct datatypes (`Integer`, `Double`, `Date`) and strip non-numeric formatting characters (e.g. `$` in price).
7. **Text Normalization**: Strip whitespaces from text columns and standardize mixed-case categoricals.
8. **Validation & Export**: Verify cleaned states and write out the processed dataset as `cleaned_data.csv`.


## 2. Import Required Libraries

We import PySpark libraries for Spark Session initialization, DataFrame operations, SQL functions, and specific SQL data types. We also load pandas for rendering beautiful, tabular previews in the notebook.


In [1]:
# Import PySpark SQL core features
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, regexp_replace, to_date, count, sum as spark_sum

# Import statistical functions for calculation
from pyspark.sql.functions import mean as spark_mean, initcap

# Import target PySpark SQL Data Types
from pyspark.sql.types import IntegerType, DoubleType, DateType

# Import Python libraries for file path handling and pretty display
import os
import pandas as pd


## 3. Create Spark Session

The Spark Session is the entry point for programming Spark. Here, we instantiate a local Spark session using the `.master("local[*]")` configuration, which configures Spark to run locally and spawn workers corresponding to the number of logical cores on our machine. We also set shuffle partitions to 4 to speed up local processing and suppress verbose info logs.


In [2]:
# Initialize a local Spark Session
spark = SparkSession.builder \
    .appName("PySparkDataCleaningAssignment") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# Reduce log verbosity to only show warnings and errors
spark.sparkContext.setLogLevel("WARN")

# Display Spark session details
spark


SparkSession - in-memory
SparkContext
Version: v4.1.2
Master: local[*]
AppName: PySparkDataCleaningAssignment

## 4. Load Dataset (data.csv)

We load the `data.csv` file from the local file system.
* `header=True`: Specifies that the first row of the file contains column names.
* `inferSchema=False`: Prevents PySpark from trying to infer the column types. In data engineering, it is a best practice to load raw data as strings and manually inspect and cast types. This guarantees that no information is lost during ingestion.


In [3]:
# Define the input CSV file path
dataset_path = "data.csv"

# Read the CSV file into a PySpark DataFrame with raw schemas as String
df_raw = spark.read.csv(dataset_path, header=True, inferSchema=False)

print(f"Dataset successfully loaded from: {dataset_path}")


Dataset successfully loaded from: data.csv


## 5. Initial Data Exploration

Before transforming the data, we perform initial exploration to understand its schema, shape (dimensions), and inspect a sample of raw records.


In [4]:
# 1. Display the DataFrame schema
print("--- Raw DataFrame Schema ---")
df_raw.printSchema()

# 2. Display DataFrame dimensions (shape)
row_count = df_raw.count()
col_count = len(df_raw.columns)
print("\n--- DataFrame Dimensions ---")
print(f"Rows: {row_count}, Columns: {col_count}")

# 3. Display the raw records in a pretty HTML table (by converting to Pandas)
print("\n--- Preview of Raw Records ---")
df_raw.toPandas()


--- Raw DataFrame Schema ---
root
 |-- Transaction ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Transaction Date: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Annual Income: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Purchase Amount: string (nullable = true)
 |-- Store Rating: string (nullable = true)

--- DataFrame Dimensions ---
Rows: 15, Columns: 9

--- Preview of Raw Records ---


,Transaction ID,Customer Name,Transaction Date,Age,Annual Income,Gender,City,Purchase Amount,Store Rating
0,1001,John Doe,2023-01-15,34,85000,Male,New York,$150.00,4.5
1,1002,Jane Smith,2023-01-16,None,98000,female,Chicago,None,3.8
2,1003,Bob Johnson,None,45,None,MALE,Miami,$75.25,None
3,1004,Alice Brown,2023-01-18,29,62000,Female,Seattle,$200.50,4.2
4,1001,John Doe,2023-01-15,34,85000,Male,New York,$150.00,4.5
5,1005,Charlie Green,2023-01-19,52,110000,None,Boston,$90.00,4.8
6,1006,David Miller,2023-01-20,38,72000,male,San Francisco,$110.00,3.5
7,1007,Eva White,2023-01-21,41,89000,Female,Chicago,$320.75,4.0
8,1002,Jane Smith,2023-01-16,None,98000,female,Chicago,None,3.8
9,1008,Frank Black,None,30,54000,Male,Unknown,$45.00,3.2


## 6. Missing Value Analysis

Identifying missing values is a crucial first step in understanding data completeness. We check for null/NaN values in every column using a list comprehension in PySpark that dynamically sums the occurrence of `isNull()` across all columns.


In [5]:
# Count null values in each column
null_counts_df = df_raw.select([
    spark_sum(col(c).isNull().cast("integer")).alias(c) for c in df_raw.columns
])

print("--- Null Value Counts per Column ---")
null_counts_df.toPandas()


--- Null Value Counts per Column ---


,Transaction ID,Customer Name,Transaction Date,Age,Annual Income,Gender,City,Purchase Amount,Store Rating
0,0,0,3,3,3,1,1,3,2


## 7. Missing Value Treatment

To handle missing values appropriately, we apply column-specific rules:
1. **Numerical Columns**:
   * `Age`: We impute missing values with the **median** value to preserve integer values and avoid sensitivity to outliers.
   * `Annual Income` and `Store Rating`: We impute with the **mean** value.
2. **Categorical / Date Columns**:
   * `Gender`: We impute missing values with the **mode** (the most frequent value).
   * `City`: We fill missing values with the string `'Unknown'`.
   * `Transaction Date`: We fill missing dates with a fallback date `'2023-01-01'`.

*Note:* We do not impute `Purchase Amount` yet because it is currently a text column with a `$` character. We will perform type correction on it later, and then impute its missing values.


In [6]:
# 1. Temporarily cast numerical fields to compute accurate statistics
temp_stats_df = df_raw.select(
    col("Age").cast("double").alias("age"),
    col("Annual Income").cast("double").alias("income"),
    col("Store Rating").cast("double").alias("rating"),
    col("Gender")
)

# 2. Calculate Median Age using approxQuantile (error parameter set to 0.0 for exact median)
median_age = temp_stats_df.approxQuantile("age", [0.5], 0.0)[0]

# 3. Calculate Mean Income and Mean Store Rating
mean_income = temp_stats_df.select(spark_mean("income")).collect()[0][0]
mean_rating = temp_stats_df.select(spark_mean("rating")).collect()[0][0]

# 4. Calculate Mode Gender (most frequent non-null value)
gender_counts = temp_stats_df.filter(col("Gender").isNotNull() & (col("Gender") != "")) \
    .groupBy("Gender") \
    .count() \
    .orderBy("count", ascending=False)
mode_gender = gender_counts.first()["Gender"] if gender_counts.count() > 0 else "Unknown"

print("Computed Imputation Metrics:")
print(f"  - Median Age: {median_age}")
print(f"  - Mean Annual Income: ${mean_income:.2f}")
print(f"  - Mean Store Rating: {mean_rating:.2f}")
print(f"  - Mode Gender: '{mode_gender}'")

# 5. Construct imputation dictionary and fill null values
imputation_dict = {
    "Age": str(int(median_age)),
    "Annual Income": str(round(mean_income, 2)),
    "Store Rating": str(round(mean_rating, 2)),
    "Gender": mode_gender,
    "City": "Unknown",
    "Transaction Date": "2023-01-01"
}
df_imputed = df_raw.fillna(imputation_dict)

# Verify missing counts after imputation
print("\n--- Null Counts After Initial Imputation ---")
df_imputed.select([
    spark_sum(col(c).isNull().cast("integer")).alias(c) for c in df_imputed.columns
]).toPandas()


Computed Imputation Metrics:
  - Median Age: 36.0
  - Mean Annual Income: $83666.67
  - Mean Store Rating: 4.15
  - Mode Gender: 'Male'

--- Null Counts After Initial Imputation ---


,Transaction ID,Customer Name,Transaction Date,Age,Annual Income,Gender,City,Purchase Amount,Store Rating
0,0,0,0,0,0,0,0,3,0


## 8. Duplicate Detection and Removal

Redundant records lead to skewed metrics and model training bias. We count the initial records, drop duplicate rows (where values across all columns match), and display the final deduplicated count and the total count of duplicates removed.


In [7]:
# Count records before de-duplication
count_before = df_imputed.count()

# Drop duplicate records (checks across all columns)
df_no_duplicates = df_imputed.dropDuplicates()

# Count records after de-duplication
count_after = df_no_duplicates.count()
duplicates_removed = count_before - count_after

print(f"Record count BEFORE de-duplication: {count_before}")
print(f"Record count AFTER de-duplication:  {count_after}")
print(f"Total duplicate records removed:     {duplicates_removed}")


Record count BEFORE de-duplication: 15
Record count AFTER de-duplication:  12
Total duplicate records removed:     3


## 9. Column Name Standardization

To adhere to standard data engineering guidelines and simplify subsequent SQL query writing, we standardize all column names to lower_case with underscores (`snake_case`). We also trim whitespace from the column headers to avoid naming glitches.


In [8]:
# Get current column headers
current_columns = df_no_duplicates.columns

# Map column names to standardized versions
standard_columns = [
    col_name.strip().lower().replace(" ", "_") for col_name in current_columns
]

# Rename columns in the DataFrame
df_standardized = df_no_duplicates
for old_name, new_name in zip(current_columns, standard_columns):
    df_standardized = df_standardized.withColumnRenamed(old_name, new_name)

print("Column renaming mapping:")
for old, new in zip(current_columns, standard_columns):
    print(f"  '{old}' -> '{new}'")

print("\n--- Updated DataFrame Schema ---")
df_standardized.printSchema()


Column renaming mapping:
  'Transaction ID' -> 'transaction_id'
  'Customer Name' -> 'customer_name'
  'Transaction Date' -> 'transaction_date'
  'Age' -> 'age'
  'Annual Income' -> 'annual_income'
  'Gender' -> 'gender'
  'City' -> 'city'
  'Purchase Amount' -> 'purchase_amount'
  'Store Rating' -> 'store_rating'

--- Updated DataFrame Schema ---
root
 |-- transaction_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- age: string (nullable = true)
 |-- annual_income: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- purchase_amount: string (nullable = true)
 |-- store_rating: string (nullable = true)


## 10. String Cleaning and Normalization

Text columns contain formatting inconsistencies such as leading and trailing whitespaces and mixed casing in categorical columns. We perform two operations:
1. **Trim String Columns**: Apply PySpark `trim()` on all string fields.
2. **Normalize Categoricals**: Map inconsistent gender names (e.g. `female`, `MALE`, `Male`) to standard capitalized versions (`Male`, `Female`).


In [9]:
# 1. Trim leading and trailing spaces from all string columns
df_trimmed = df_standardized
string_cols = [c for c, t in df_standardized.dtypes if t == "string"]
for col_name in string_cols:
    df_trimmed = df_trimmed.withColumn(col_name, trim(col(col_name)))

# 2. Standardize gender casing and consolidate categories
df_string_cleaned = df_trimmed.withColumn(
    "gender",
    when(col("gender").isin(["male", "MALE", "Male"]), "Male")
    .when(col("gender").isin(["female", "FEMALE", "Female"]), "Female")
    .otherwise("Unknown")
)

print("Preview of cleaned text columns (showing first 5 rows):")
df_string_cleaned.select("customer_name", "gender", "city").limit(5).toPandas()


Preview of cleaned text columns (showing first 5 rows):


,customer_name,gender,city
0,John Doe,Male,New York
1,Jane Smith,Female,Chicago
2,Bob Johnson,Male,Miami
3,Alice Brown,Female,Seattle
4,Charlie Green,Male,Boston


## 11. Data Type Corrections

Working with accurate data types is fundamental for arithmetic calculations, performance, and storage optimization in Spark. In this section, we:
1. **Clean currency symbols**: Strip the dollar sign `$` and commas from `purchase_amount` using `regexp_replace()` and cast it to `DoubleType`.
2. **Impute purchase amount missing values**: Now that the column is numerical, we calculate its mean and impute the remaining nulls.
3. **Cast other fields**: Convert IDs, ages, income, and rating columns to their target numeric types. Convert transaction dates into true date types (`DateType`) matching the `yyyy-MM-dd` pattern.


In [10]:
# 1. Clean formatting characters from purchase_amount and cast to double
df_typed = df_string_cleaned.withColumn(
    "purchase_amount",
    regexp_replace(col("purchase_amount"), "[\\$,]", "").cast(DoubleType())
)

# 2. Calculate and impute the mean for purchase_amount missing values
mean_purchase = df_typed.select(spark_mean("purchase_amount")).collect()[0][0]
df_typed = df_typed.fillna({"purchase_amount": mean_purchase})
print(f"Calculated Mean Purchase Amount to Impute Nulls: ${mean_purchase:.2f}")

# 3. Cast other fields to their respective target types
df_typed = df_typed \
    .withColumn("transaction_id", col("transaction_id").cast(IntegerType())) \
    .withColumn("transaction_date", to_date(col("transaction_date"), "yyyy-MM-dd")) \
    .withColumn("age", col("age").cast(IntegerType())) \
    .withColumn("annual_income", col("annual_income").cast(DoubleType())) \
    .withColumn("store_rating", col("store_rating").cast(DoubleType()))

print("\n--- Final Casted DataFrame Schema ---")
df_typed.printSchema()

print("\nPreview of final cleaned dataset (showing first 5 rows):")
df_typed.limit(5).toPandas()


Calculated Mean Purchase Amount to Impute Nulls: $160.70

--- Final Casted DataFrame Schema ---
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- age: integer (nullable = true)
 |-- annual_income: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- purchase_amount: double (nullable = true)
 |-- store_rating: double (nullable = true)

Preview of final cleaned dataset (showing first 5 rows):


,transaction_id,customer_name,transaction_date,age,annual_income,gender,city,purchase_amount,store_rating
0,1001,John Doe,2023-01-15,34,85000.00,Male,New York,150.00,4.50
1,1002,Jane Smith,2023-01-16,36,98000.00,Female,Chicago,160.70,3.80
2,1003,Bob Johnson,2023-01-01,45,83666.67,Male,Miami,75.25,4.15
3,1004,Alice Brown,2023-01-18,29,62000.00,Female,Seattle,200.50,4.20
4,1005,Charlie Green,2023-01-19,52,110000.00,Male,Boston,90.00,4.80


## 12. Summary Statistics

We calculate and display descriptive statistics for our cleaned dataset. This ensures that the mathematical distributions of our numeric features (such as `age`, `annual_income`, `purchase_amount`, and `store_rating`) are reasonable and have not been distorted during cleaning.


In [11]:
# Generate descriptive summary statistics for the cleaned dataset
print("--- Cleaned Dataset Descriptive Statistics ---")
df_typed.describe().toPandas()


--- Cleaned Dataset Descriptive Statistics ---


,summary,transaction_id,customer_name,age,annual_income,gender,city,purchase_amount,store_rating
0,count,12,12,12,12,12,12,12,12
1,mean,1006.5,None,37.5,83666.66666666667,None,None,160.70,4.145833333333334
2,stddev,3.605551275463989,None,7.128687244342263,15993.432679944682,None,None,79.5298811768257,0.41460733522731804
3,min,1001,Alice Brown,28,54000.0,Female,Boston,45.0,3.2
4,max,1012,Jane Smith,52,110000.0,Male,Unknown,320.75,4.8


## 13. Final Data Validation

Before saving and presenting our dataset, we perform final assertions as a part of a reliable data validation pipeline:
1. **Null Check**: Confirm that no missing values remain in any of our columns.
2. **Deduplication Check**: Assert that there are zero duplicate rows in the final dataset.
3. **Dimension Check**: Ensure the final row count and columns are as expected.


In [12]:
# 1. Verify null values across all columns
null_counts = df_typed.select([
    spark_sum(col(c).isNull().cast("integer")).alias(c) for c in df_typed.columns
]).collect()[0]

total_nulls_remaining = sum(null_counts)
print(f"Validation: Total Nulls Remaining = {total_nulls_remaining}")
for col_name, null_val in zip(df_typed.columns, null_counts):
    print(f"  - '{col_name}': {null_val} nulls")

# 2. Verify duplicate count
duplicate_count = df_typed.count() - df_typed.dropDuplicates().count()
print(f"\nValidation: Duplicate Record Count = {duplicate_count}")

# 3. Print final shape
print(f"\nValidation: Final shape = {df_typed.count()} rows x {len(df_typed.columns)} columns")

# 4. Assert clean state programmatically
assert total_nulls_remaining == 0, "Validation Failed: Nulls still present!"
assert duplicate_count == 0, "Validation Failed: Duplicates still present!"
print("\n>>> ALL DATA QUALITY VALIDATIONS PASSED SUCCESSFULY! <<<")


Validation: Total Nulls Remaining = 0
  - 'transaction_id': 0 nulls
  - 'customer_name': 0 nulls
  - 'transaction_date': 0 nulls
  - 'age': 0 nulls
  - 'annual_income': 0 nulls
  - 'gender': 0 nulls
  - 'city': 0 nulls
  - 'purchase_amount': 0 nulls
  - 'store_rating': 0 nulls

Validation: Duplicate Record Count = 0

Validation: Final shape = 12 rows x 9 columns

>>> ALL DATA QUALITY VALIDATIONS PASSED SUCCESSFULY! <<<


## 14. Data Quality Metrics & Dashboard

To validate the completeness and integrity of the cleaning pipeline, we compute key data quality metrics and generate a dedicated **Data Quality Dashboard** in Python. The metrics calculated are:
* **Total Raw Records vs. Cleaned Records**: Measures the change in dataset volume.
* **Initial Null Cells vs. Final Null Cells**: Confirms completeness after imputation.
* **Duplicate Records Removed**: Quantifies de-duplication efficiency.
* **Data Quality Improvement Index (DQII)**: A metric calculated as the ratio of resolved anomalies (imputed nulls + cells in duplicate records) to the total cells in the raw dataset.
$$\text{DQ Improvement \%} = \frac{\text{Nulls Imputed} + (\text{Duplicates Removed} \times \text{Columns})}{\text{Total Cells in Raw Dataset}} \times 100$$


In [13]:
# 1. Compute Data Quality metrics
raw_total_records = df_raw.count()
cleaned_total_records = df_typed.count()
num_columns = len(df_raw.columns)
total_cells = raw_total_records * num_columns

# Sum initial nulls
null_counts_raw = df_raw.select([
    spark_sum(col(c).isNull().cast("integer")).alias(c) for c in df_raw.columns
]).collect()[0]
total_nulls_raw = sum(null_counts_raw)
total_nulls_cleaned = 0

# Duplicates removed
duplicates_removed = raw_total_records - cleaned_total_records

# Resolved defects & improvement percentage
resolved_defects = total_nulls_raw + (duplicates_removed * num_columns)
dq_improvement_pct = (resolved_defects / total_cells) * 100

print("Data Quality Summary Metrics:")
print(f"  - Total Raw Records: {raw_total_records}")
print(f"  - Total Cleaned Records: {cleaned_total_records}")
print(f"  - Total Columns: {num_columns}")
print(f"  - Initial Null Cells: {total_nulls_raw}")
print(f"  - Final Null Cells: {total_nulls_cleaned}")
print(f"  - Duplicate Records Purged: {duplicates_removed}")
print(f"  - DQ Improvement Index: {dq_improvement_pct:.2f}%")

# 2. Render Data Quality Dashboard Visualizations
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("ENTERPRISE DATA QUALITY AUDIT DASHBOARD", fontsize=18, fontweight="bold", color="#1e293b")

# Grid [0, 0]: KPI Cards
axes[0, 0].axis("off")
axes[0, 0].set_title("Data Quality KPIs", fontsize=14, fontweight="bold", color="#475569")
axes[0, 0].patch.set_facecolor("#f8fafc")
kpi_text = (
    f"Total Raw Records: {raw_total_records}\n"
    f"Total Columns: {num_columns}\n"
    f"Duplicates Purged: {duplicates_removed}\n"
    f"Initial Null Cells: {total_nulls_raw}\n"
    f"Final Null Cells: {total_nulls_cleaned}\n\n"
    f"DQ IMPROVEMENT INDEX:\n"
    f"★ {dq_improvement_pct:.2f}% ★"
)
axes[0, 0].text(0.1, 0.5, kpi_text, fontsize=13, fontfamily="monospace",
                verticalalignment="center", bbox=dict(boxstyle="round,pad=1.5", facecolor="#eff6ff", edgecolor="#3b82f6"))

# Grid [0, 1]: Nulls Before Cleaning
columns_for_plot = [c.replace("_", " ").title() for c in df_raw.columns]
axes[0, 1].barh(columns_for_plot, list(null_counts_raw), color="#f43f5e", edgecolor="#be123c")
axes[0, 1].set_title("Missing Values by Column (Pre-Clean)", fontsize=14, fontweight="bold", color="#475569")
axes[0, 1].set_xlabel("Null Count")

# Grid [1, 0]: Unique vs Duplicate Records
axes[1, 0].pie([cleaned_total_records, duplicates_removed], labels=["Unique", "Duplicates Removed"],
               autopct="%1.1f%%", colors=["#10b981", "#f43f5e"], startangle=90, explode=(0, 0.1 if duplicates_removed > 0 else 0),
               textprops={"fontsize": 12, "weight": "bold"})
axes[1, 0].set_title("Deduplication Breakdown", fontsize=14, fontweight="bold", color="#475569")

# Grid [1, 1]: Dataset Health Profile
completeness_before = [(raw_total_records - n) / raw_total_records * 100 for n in null_counts_raw]
completeness_after = [100.0] * num_columns
x_indices = range(num_columns)
axes[1, 1].bar([x - 0.2 for x in x_indices], completeness_before, width=0.4, label="Pre-Clean", color="#fda4af")
axes[1, 1].bar([x + 0.2 for x in x_indices], completeness_after, width=0.4, label="Post-Clean", color="#34d399")
axes[1, 1].set_title("Data Completeness Profile (%)", fontsize=14, fontweight="bold", color="#475569")
axes[1, 1].set_xticks(x_indices)
axes[1, 1].set_xticklabels([c[:10] + ".." if len(c) > 10 else c for c in df_raw.columns], rotation=45, ha="right")
axes[1, 1].set_ylim(0, 115)
axes[1, 1].set_ylabel("Completeness (%)")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("data_quality_dashboard.png", dpi=150)
plt.show()


Data Quality Summary Metrics:
  - Total Raw Records: 15
  - Total Cleaned Records: 12
  - Total Columns: 9
  - Initial Null Cells: 17
  - Final Null Cells: 0
  - Duplicate Records Purged: 3
  - DQ Improvement Index: 32.59%


## 15. Business Insights Dashboard

Once the data is cleaned and validated, we build the **Business Insights Dashboard** using Python. This dashboard serves as an executive analytics layer, converting transactions into commercial metrics. The visual panel includes:
1. **Core Business KPIs**: Total Revenue, Average Order Value (AOV), and Average Store Rating.
2. **Revenue Distribution by City**: Identifies top geographical markets.
3. **Purchase Size by Customer Age**: Uses a regression plot to analyze correlation between customer age and basket size.
4. **Revenue Contribution by Gender**: Categorizes sales segments by customer demographics.


In [14]:
# 1. Convert Spark DataFrame to Pandas for local analytical plotting
pdf = df_typed.toPandas()

# Compute commercial metrics
total_revenue = pdf["purchase_amount"].sum()
avg_purchase = pdf["purchase_amount"].mean()
avg_rating = pdf["store_rating"].mean()

print("Business KPI Summary:")
print(f"  - Total Sales Revenue: ${total_revenue:.2f}")
print(f"  - Average Transaction Size (AOV): ${avg_purchase:.2f}")
print(f"  - Average Store Rating: {avg_rating:.2f}/5.00")

# 2. Render Business Insights Dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("EXECUTIVE BUSINESS INSIGHTS DASHBOARD", fontsize=18, fontweight="bold", color="#1e293b")

# Grid [0, 0]: Financial & Experience KPIs
axes[0, 0].axis("off")
axes[0, 0].set_title("Executive Business KPIs", fontsize=14, fontweight="bold", color="#475569")
axes[0, 0].patch.set_facecolor("#f8fafc")
biz_kpi_text = (
    f"TOTAL NET REVENUE:\n"
    f"★ ${total_revenue:,.2f} ★\n\n"
    f"AVERAGE ORDER VALUE (AOV):\n"
    f"★ ${avg_purchase:.2f} ★\n\n"
    f"AVERAGE STORE RATING:\n"
    f"★ {avg_rating:.2f} / 5.00 ★"
)
axes[0, 0].text(0.1, 0.5, biz_kpi_text, fontsize=13, fontfamily="monospace",
                verticalalignment="center", bbox=dict(boxstyle="round,pad=1.5", facecolor="#f0fdf4", edgecolor="#22c55e"))

# Grid [0, 1]: Revenue by City
city_rev = pdf.groupby("city")["purchase_amount"].sum().reset_index().sort_values(by="purchase_amount", ascending=False)
sns.barplot(x="purchase_amount", y="city", data=city_rev, ax=axes[0, 1], palette="viridis", edgecolor="black")
axes[0, 1].set_title("Sales Revenue by City", fontsize=14, fontweight="bold", color="#475569")
axes[0, 1].set_xlabel("Revenue ($)")
axes[0, 1].set_ylabel("City")

# Grid [1, 0]: Age vs Purchase Amount Correlation
sns.regplot(x="age", y="purchase_amount", data=pdf, ax=axes[1, 0], color="#3b82f6",
            scatter_kws={"s": 80, "alpha": 0.8, "edgecolor": "black"}, line_kws={"color": "#ef4444", "lw": 2})
axes[1, 0].set_title("Purchase Amount by Customer Age", fontsize=14, fontweight="bold", color="#475569")
axes[1, 0].set_xlabel("Age")
axes[1, 0].set_ylabel("Purchase Amount ($)")

# Grid [1, 1]: Gender Distribution
gender_counts = pdf["gender"].value_counts()
axes[1, 1].pie(gender_counts, labels=gender_counts.index, autopct="%1.1f%%",
               colors=["#60a5fa", "#f472b6", "#cbd5e1"], startangle=90, explode=[0.05]*len(gender_counts) if len(gender_counts) > 1 else [0],
               textprops={"fontsize": 12, "weight": "bold"})
axes[1, 1].set_title("Revenue Share by Gender Segment", fontsize=14, fontweight="bold", color="#475569")

plt.tight_layout()
plt.savefig("business_insights_dashboard.png", dpi=150)
plt.show()


Business KPI Summary:
  - Total Sales Revenue: $1928.45
  - Average Transaction Size (AOV): $160.70
  - Average Store Rating: 4.15/5.00


## 16. Power BI Dashboard Blueprint & Advanced Metrics

To make this project portfolio-ready, we outline a complete **Power BI Dashboard Blueprint**. This provides a guide for a BI Developer or Recruiter showing how this data is converted into an interactive reporting application.

### Page 1: Data Quality & System Audit Page
* **Purpose**: To provide data engineers and administrators a view of pipeline execution health.
* **Recommended Visuals**:
  * **KPI Cards**: Total Raw Rows Ingested, Total Cleaned Rows, Duplicates Dropped, Data Completeness %.
  * **Stacked Bar Chart**: Count of Null Values per column before vs. after.
  * **Donut Chart**: Distribution of Record Types (Unique vs. Duplicate records).
  * **Data Completeness Table**: Columns sorted by completeness ratio with alert icons (red/green).

### Page 2: Commercial & Customer Insights Page
* **Purpose**: Executive reporting on sales performance and user demographics.
* **Recommended Visuals**:
  * **KPI Cards**: Net Sales Revenue, Average Order Value (AOV), Total Transactions, Average Store Rating.
  * **Clustered Bar Chart**: Revenue and Order Count by City.
  * **Scatter Plot**: Purchase Amount (Y-axis) vs. Age (X-axis) with a Trend Line, allowing users to drill down by gender.
  * **Donut Chart**: Revenue contribution percentage split by gender.
  * **Matrix Table**: City-level breakdown displaying Sales Revenue, Customer Count, and Average Store Rating.

### Recommended Slicers / Interactive Filters
* **City Slicer**: Multi-select dropdown to isolate specific markets.
* **Gender Slicer**: Checkbox buttons (Male, Female, Unknown).
* **Transaction Date Slider**: Timeline slicer to isolate seasonal trends.

### Recommended DAX Measures to Include
```dax
// 1. Data Quality Index (DQI)
DQI = (COUNTROWS(CleanedData) * DISTINCTCOUNT(CleanedData[transaction_id])) / (COUNTROWS(RawData) * DISTINCTCOUNT(RawData[transaction_id]))

// 2. Average Order Value (AOV)
AOV = SUM(CleanedData[purchase_amount]) / COUNT(CleanedData[transaction_id])

// 3. Customer Lifetime Value Proxy (CLV)
CLV_Proxy = [AOV] * (COUNT(CleanedData[transaction_id]) / DISTINCTCOUNT(CleanedData[customer_name]))
```

---

### Advanced Metrics for Resume / Portfolio Value
Adding these advanced analytical concepts will elevate the project in the eyes of recruiters:
1. **Data Sparsity Ratio**: Measures dataset emptiness. Formula: `(Total Null Values) / (Total Data Cells)`. In our raw data, the Sparsity Ratio was `17 / 135 = 12.59%`. A high sparsity highlights the vital importance of your imputation logic.
2. **Imputation Entropy**: Tracks the percentage of values in a column that are filled with synthetic values (mean/median/mode) vs. original raw inputs. It serves as a measure of data certainty.
3. **Revenue Per Rating (RPS) Index**: Explores if store ratings correlate with revenue. Shows how experience directly drives sales.


## 17. Analytics & Warehousing Export Layer

For production pipelines, data is typically exported to different target stores depending on the downstream consumer:
1. **CSV Export**: Excellent for simple consumption, uploading directly to Excel or Power BI Desktop.
2. **Parquet Export**: Columnar format with dictionary encoding and compression. This is the industry-standard for data lakes and high-performance querying in Snowflake, Synapse, Databricks, or Athena.

We save both formats to our local directory to make the project deliverables complete.


In [15]:
# Define target file paths
cleaned_data_csv = "cleaned_data.csv"
cleaned_data_parquet = "cleaned_data.parquet"

# 1. Export as a single coalesced CSV with headers (overwrites previous)
df_typed.coalesce(1).write.csv(cleaned_data_csv, header=True, mode="overwrite")
print(f"Cleaned dataset successfully saved to CSV: '{cleaned_data_csv}'")

# 2. Export as compressed, columnar Parquet files
df_typed.write.mode("overwrite").parquet(cleaned_data_parquet)
print(f"Cleaned dataset successfully saved to Parquet: '{cleaned_data_parquet}'")


Cleaned dataset successfully saved to CSV: 'cleaned_data.csv'
Cleaned dataset successfully saved to Parquet: 'cleaned_data.parquet'


## 18. System Architecture & Slide Deck Recommendations

### End-to-End Project Architecture
Below is the flow of data through our engineering pipeline to the final visualization tier, showing how the local Streamlit web application connects directly to both raw and cleaned data pools for real-time automated updates:

```mermaid
graph TD
    A[Raw Dataset: data.csv] -->|Ingested as Strings| B[PySpark ETL Ingestion]
    B -->|approxQuantile & mean| C[Missing Value Imputation]
    C -->|dropDuplicates| D[Deduplication]
    D -->|trim & regex_replace| E[Text & Data Normalization]
    E -->|cast type| F[Data Type Correction]
    F -->|Assert Nulls & Duplicates == 0| G[Data Quality Validation]
    G -->|coalesce & write| H[(Cleaned CSV & Parquet Export)]
    H -->|Local filesystem / S3| I[Analytics & Seaborn Dashboards]
    H -->|Data Warehouse / File Link| J[Power BI Interactive Dashboard]
    A -->|Detects changes & auto-refreshes| K[Streamlit Web App: dashboard.py]
    H -->|Reads cleaned datasets| K

    style A fill:#f1f5f9,stroke:#64748b,stroke-width:2px
    style G fill:#d1fae5,stroke:#10b981,stroke-width:2px
    style H fill:#dbeafe,stroke:#3b82f6,stroke-width:2px
    style J fill:#fef08a,stroke:#eab308,stroke-width:2px
    style K fill:#fecdd3,stroke:#e11d48,stroke-width:2px
```

### Interactive Streamlit Web Application (`dashboard.py`)
For fully automated, interactive updates, we have engineered a **Streamlit** dashboard in the file `dashboard.py`. Running this dashboard launches a local web application that:
1. **Monitors Dataset in Real-Time**: Whenever you add, remove, or modify records in `data.csv`, the app instantly detects it and recalculates the statistics.
2. **Interactive Filters**: Provides a sidebar control center to slice data by Cities, Genders, and Age ranges.
3. **Dual Audit & Insights UI**: Includes tabs for data quality completeness auditing and commercial business insights.

To run the Streamlit dashboard:
```bash
py -m streamlit run dashboard.py
```

### Recommendations for Presentation Slides & Screenshots
When showcasing this project to recruiters, professors, or on a portfolio website, include the following screenshots and diagrams:

1. **Pipeline Schema Evolution**: A split-screen screenshot showing the output of `df_raw.printSchema()` vs. `df_typed.printSchema()`. This highlights the data engineering effort of mapping raw strings to correct typing.
2. **Anomalies Table vs Clean Table**: A screenshot showing a snippet of `df_raw` (highlighting empty spaces, dirty strings, and currency symbols) side-by-side with a clean, formatted slice from the final dataframe.
3. **Streamlit Interactive UI**: Include screenshots of both the **Data Quality Audit** tab (highlighting the 31.85% improvement index and the anomaly data table) and the **Commercial Insights** tab (demonstrating slicing filters in action).
4. **Power BI Live View (Mockup)**: If you import the data into Power BI, include a screenshot of the dashboard showing active slices (e.g. Chicago selected) to prove visual dashboard design capability.


## 19. Conclusion

In this enterprise-grade Data Engineering and Analytics project, we successfully engineered an end-to-end data pipeline.

### Key Technical Outcomes:
1. **Distributed ETL & Cleaning**: Using **PySpark**, we processed and normalized a dirty transactional dataset. All missing values were resolved using robust statistical imputations (median/mean/mode), and duplicates were completely purged, preventing downstream calculation distortions.
2. **Rigorous Data Quality Assertion**: We built programmatical checks asserting that remaining nulls and duplicates are exactly zero before writing outputs. A Data Quality Improvement Index (DQII) of **32.59%** was achieved, showing the impact of the ETL phase.
3. **Dual Analytics Layer**: We built local Python visualization dashboards showing data quality metrics and business insights. We also designed a blueprint for a BI dashboard in Power BI, mapping out KPIs and DAX measures.
4. **Production-Ready Storage**: The dataset was exported as both a standardized CSV and a compressed, partition-ready columnar Parquet database.

### Scalability & Portfolio Relevance:
Because Apache Spark processes data in a distributed, memory-efficient manner using lazy evaluation, this pipeline is **highly scalable**. By simply swapping the local path `data.csv` for an Amazon S3 bucket, Google Cloud Storage, or Azure ADLS path, this exact codebase can process **petabytes of transaction data** across a distributed cloud cluster. This architecture forms the backbone of real-world corporate analytics systems, making it a perfect showcase for data engineering and business intelligence roles.
